## Exercises on Continuous Control

These paper-and-pencil exercises reinforce Chapter 10: Gaussian policies and their score function, the $\tanh$ squashing and its change-of-variables correction, the deterministic policy gradient as a chain rule, the DDPG target with soft (Polyak) target updates, the downward bias of clipped double Q-learning, target policy smoothing, the maximum-entropy objective and the soft Bellman backup, the automatic temperature, and the reparameterisation trick. Notation: actor $\mu_\theta(s)$ (deterministic) or $\pi_\theta(a\mid s)$ (stochastic), critic $Q_\phi(s,a)$ with target copy $Q_{\phi^-}$, soft-update factor $\tau$, temperature $\alpha$ (not a step size in this chapter), target entropy $\bar H$, action bound $a_{\max}$.

### Exercise 10.1 — The score function of a Gaussian policy

A one-dimensional Gaussian policy has mean $\mu=0.5$ and standard deviation $\sigma=0.4$. The agent takes the action $a=1.1$ and observes an advantage $A=+2$.

1. Compute $\log\pi(a\mid s)$.
2. Compute the score components $\partial\log\pi/\partial\mu$ and $\partial\log\pi/\partial\sigma$.
3. Give the direction of the policy-gradient update of $\mu$ and $\sigma$ (sign only) and interpret it.

**Step 1 — Log-density.** $\ \log\pi(a\mid s) = -\dfrac{(a-\mu)^2}{2\sigma^2} - \log\sigma - \tfrac12\log 2\pi = -\dfrac{0.36}{0.32} - \log 0.4 - 0.9189 = -1.125 + 0.9163 - 0.9189 = -1.128.$

**Step 2 — Score.**

$\displaystyle \frac{\partial\log\pi}{\partial\mu} = \frac{a-\mu}{\sigma^2} = \frac{0.6}{0.16} = 3.75, \qquad \frac{\partial\log\pi}{\partial\sigma} = \frac{(a-\mu)^2-\sigma^2}{\sigma^3} = \frac{0.36-0.16}{0.064} = 3.125.$

**Step 3 — Update direction.** The update is $\Delta\theta \propto A\,\nabla_\theta\log\pi$ with $A=+2>0$, so both components are increased: $\mu$ moves **towards** the action taken ($a>\mu$, score positive), and $\sigma$ **grows**, because the good action was further from the mean than one standard deviation ($|a-\mu|=0.6>\sigma=0.4$). Had the action been within one standard deviation, the same positive advantage would have *shrunk* $\sigma$.

**Key concept**

For a Gaussian policy the score has a mechanical reading: $(a-\mu)/\sigma^2$ pulls the mean towards good actions, and $\big((a-\mu)^2-\sigma^2\big)/\sigma^3$ widens the distribution when good actions are found far from the mean and narrows it when they are found close. The policy gradient methods of Chapters 08–09 apply unchanged once $\log\pi$ is replaced by this expression.

### Exercise 10.2 — Squashing with $\tanh$: the change-of-variables correction

A squashed Gaussian policy draws $u\sim\mathcal N(\mu,\sigma^2)$ and outputs $a=a_{\max}\tanh u$ with $a_{\max}=2$.

1. Derive $\log\pi(a\mid s)$ from $\log\mathcal N(u;\mu,\sigma^2)$.
2. Evaluate the correction term for $u=0$ and for $u=2$. Where is it large, and why does it matter for the entropy term of SAC?

**Step 1 — Change of variables.** For a monotone map $a=g(u)$ the densities satisfy $\pi(a) = p(u)\,\big|\mathrm{d}u/\mathrm{d}a\big| = p(u)\big/\big|g'(u)\big|$. Here $g'(u) = a_{\max}(1-\tanh^2u)$, so

$\displaystyle \log\pi(a\mid s) = \log\mathcal N(u;\mu,\sigma^2) - \log\big(a_{\max}\,(1-\tanh^2 u)\big).$

(In $d$ dimensions the correction is summed over the components.)

**Step 2 — Evaluate.**

- $u=0$: $\tanh 0 = 0$, correction $= -\log(2\cdot 1) = -0.693$.
- $u=2$: $\tanh 2 = 0.9640$, $1-\tanh^2 2 = 0.0707$, correction $= -\log(2\cdot 0.0707) = -\log 0.1413 = +1.957$.

Near the bounds the $\tanh$ saturates: a wide interval of $u$ is compressed into a narrow interval of $a$, so the density of $a$ is **much higher** than that of $u$ (the correction adds $+1.96$ nats at $u=2$). SAC's entropy term $-\alpha\log\pi(a\mid s)$ would therefore be badly overestimated at the bounds without the correction — and the bounds (full throttle) are exactly where a controller often operates.

**Key concept**

Squashing a Gaussian through $\tanh$ keeps actions inside the box but changes the density by the factor $1/\big(a_{\max}(1-\tanh^2u)\big)$. The correction is negligible in the middle of the range and large at the edges; it must be included whenever $\log\pi$ is used, as in the SAC actor loss, the soft target and the temperature update.

### Exercise 10.3 — The deterministic policy gradient as a chain rule

In a fixed state $s$ the critic is $Q(s,a) = -\,(a-3)^2 + 5$ (scalar action) and the actor is linear, $\mu_\theta(s)=\theta_1 x_1+\theta_2 x_2$ with features $x=(1,\,2)$ and parameters $\theta=(0.5,\,0.5)$.

1. Compute the current action $\mu_\theta(s)$ and $\nabla_a Q(s,a)$ at that action.
2. Compute $\nabla_\theta\mu_\theta(s)$ and the deterministic policy gradient $\nabla_\theta J = \nabla_\theta\mu_\theta(s)\,\nabla_aQ(s,a)\big|_{a=\mu_\theta(s)}$.
3. Perform one ascent step with step size $\eta=0.1$ and verify that $Q(s,\mu_\theta(s))$ increased.

**Step 1 — Action and critic gradient.** $\ \mu_\theta(s) = 0.5(1)+0.5(2) = 1.5$; $\ \nabla_aQ = -2(a-3) = -2(1.5-3) = +3$. The critic says: *increase the action*.

**Step 2 — Chain rule.** $\ \nabla_\theta\mu_\theta(s) = x = (1,2)$, so

$\displaystyle \nabla_\theta J = (1,2)\cdot 3 = (3,\,6).$

**Step 3 — Ascent step.** $\ \theta\leftarrow\theta+0.1\,(3,6) = (0.8,\,1.1)$, new action $\mu_\theta(s) = 0.8+2.2 = 3.0$, which is exactly the maximiser of $Q$ in this state. The value rose from $Q(s,1.5) = -2.25+5 = 2.75$ to $Q(s,3.0)=5$. (The step landed on the optimum only because $Q$ is quadratic and the step size happened to match; in general several steps are needed.)

**Key concept**

The deterministic policy gradient needs no sampling over actions: the critic's gradient $\nabla_aQ$ says in which direction the action should move, and $\nabla_\theta\mu$ translates that direction into parameter space. Because the expectation is only over states, the states can come from a replay memory — this is what makes DDPG off-policy.

### Exercise 10.4 — DDPG target and soft target updates

A transition $(s,a,r,s')$ has $r=-1$, $\gamma=0.99$, $d=0$. The target actor proposes $\mu_{\theta^-}(s')=0.4$, the target critic gives $Q_{\phi^-}(s',0.4)=-20$, and the online critic currently predicts $Q_\phi(s,a)=-25$.

1. Compute the DDPG target $y$ and the TD error.
2. A scalar target parameter is $\phi^-=1.0$ and its online counterpart is $\phi=2.0$. Apply the soft update with $\tau=0.005$ once, then give the value after $n$ updates (with $\phi$ fixed) and the number of updates after which the gap has shrunk by a factor $e$.
3. Compare with a hard update every $C=200$ steps.

**Step 1 — Target and TD error.** $\ y = r+\gamma(1-d)\,Q_{\phi^-}\big(s',\mu_{\theta^-}(s')\big) = -1+0.99(-20) = -20.8$; $\ \delta = y - Q_\phi(s,a) = -20.8-(-25) = +4.2.$ The target actor replaces the $\arg\max$ of DQN: no maximisation over actions is needed.

**Step 2 — Soft update.** One update: $\ \phi^-\leftarrow\tau\phi+(1-\tau)\phi^- = 0.005(2.0)+0.995(1.0) = 1.005.$ With $\phi$ fixed, the gap $\phi-\phi^-$ is multiplied by $(1-\tau)$ at every update, so after $n$ updates

$\displaystyle \phi^-_n = \phi - (1-\tau)^n(\phi-\phi^-_0) = 2 - 0.995^n,$

and the gap shrinks by a factor $e$ after $n = -1/\ln(0.995) \approx 199.5 \approx 1/\tau = 200$ updates: the target tracks the online network with time constant $1/\tau$.

**Step 3 — Hard vs. soft.** A hard update with $C=200$ copies $\phi$ into $\phi^-$ every $200$ steps: the target is *frozen* for $200$ steps, then *jumps*. The soft update with $\tau=1/200$ has the same characteristic time but moves the target a little at every step, so the critic's regression target changes smoothly. With two coupled networks (actor and critic) that must remain consistent, the absence of jumps is what matters.

**Key concept**

DDPG = DQN with the $\max_{a'}$ replaced by the target actor's action, plus **soft (Polyak) target updates** $\theta^-\leftarrow\tau\theta+(1-\tau)\theta^-$, an exponential moving average of the online parameters with time constant $1/\tau$.

### Exercise 10.5 — Why the minimum of two critics is pessimistic

Two critics estimate the same true value $q=10$ with independent errors: each estimate is $10+1$ or $10-1$ with probability $\tfrac12$.

1. Compute the expectation of a single estimate, of the **maximum** of the two (the DDPG/DQN-like situation, where the actor finds the higher estimate), and of the **minimum** of the two (clipped double Q-learning).
2. Explain why TD3 prefers the downward bias, and relate it to Double DQN.

**Step 1 — Expectations.** A single estimate is unbiased: $\mathbb E = 10$. For the pair, the four equally likely outcomes are $(11,11),(11,9),(9,11),(9,9)$:

$\displaystyle \mathbb E[\max] = \tfrac14(11+11+11+9) = 10.5, \qquad \mathbb E[\min] = \tfrac14(11+9+9+9) = 9.5.$

Selecting the larger of two noisy estimates **overestimates** by $0.5$; taking the smaller **underestimates** by the same amount.

**Step 2 — Why pessimism is safer.** The actor is trained to *maximise* the critic, so it drifts towards whatever the critic overestimates; the critic is then trained on targets built from those overestimated actions and the error can feed on itself (a loop of overestimation, not a one-off error). An underestimate cannot be exploited: the actor is merely discouraged from actions whose value is uncertain, which slows learning slightly but never produces a runaway. Double DQN also fought the maximisation bias, by *decoupling selection from evaluation*. Fujimoto and colleagues tried the actor-critic analogue (target actor selects, target critic evaluates) and found it ineffective: with soft updates the actor and its target change slowly and remain too similar to give independent estimates. TD3 therefore attacks the bias more bluntly, with the minimum of two *independently initialised* critics.

**Key concept**

$\max$ of noisy estimates is biased upwards, $\min$ downwards. In an actor–critic the actor exploits any upward bias, so **clipped double Q-learning** deliberately accepts a small downward bias in the target to break the overestimation loop.

### Exercise 10.6 — Target policy smoothing

In state $s'$ the (target) critic has a spurious narrow peak: $Q(s',a) = -5$ everywhere except $Q(s',a)=+5$ for $a\in[0.95,1.05]$. The target actor outputs $\mu_{\theta^-}(s')=1.0$, exactly at the peak. TD3 adds noise $\epsilon$ to the target action, uniformly distributed in $[-0.5,\,0.5]$ for this exercise (instead of clipped Gaussian noise), with $a_{\max}=2$.

1. Compute the TD3 target contribution $\mathbb E_\epsilon[Q(s',\mu_{\theta^-}(s')+\epsilon)]$ and compare it with the unsmoothed value $Q(s',1.0)$.
2. Explain what this does to the critic's training.

**Step 1 — Smoothed value.** The noisy target action $1.0+\epsilon$ is uniform on $[0.5,1.5]$, an interval of length $1$; it falls in the peak $[0.95,1.05]$ (length $0.1$) with probability $0.1$:

$\displaystyle \mathbb E_\epsilon\big[Q(s',1+\epsilon)\big] = 0.1(5)+0.9(-5) = -4.$

Without smoothing the target would use $Q(s',1.0)=+5$; with smoothing it uses $-4$, close to the value of the surrounding region.

**Step 2 — Effect.** The narrow peak is an artefact of function approximation (one lucky transition, or extrapolation). A deterministic target action evaluated exactly at the peak would propagate the $+5$ into the targets of all the transitions leading to $s'$, and the actor would be drawn to that action; the smoothed target averages $Q$ over a neighbourhood, so the artefact is washed out unless it is wide enough to be believable. Target smoothing is a regulariser enforcing that **similar actions should have similar values**.

**Key concept**

Adding clipped noise to the *target* action turns the bootstrapped value into a local average of $Q$ in action space. This removes the incentive for the actor to exploit sharp, unreliable peaks in the critic — the second of TD3's three fixes.

### Exercise 10.7 — Soft values and the soft Bellman backup

A one-dimensional Gaussian policy has standard deviation $\sigma=0.5$ in state $s'$ (ignore squashing), and the temperature is $\alpha=0.2$.

1. Compute the entropy $H\big(\pi(\cdot\mid s')\big) = \tfrac12\log(2\pi e\sigma^2)$.
2. A transition has $r=1$, $\gamma=0.9$, $d=0$, and for the sampled next action the two target critics give $Q_1(s',a')=4.0$, $Q_2(s',a')=4.4$ with $\log\pi(a'\mid s')=-0.4$. Compute the SAC target $y$.
3. In expectation over $a'$, what does the bootstrapped term $Q-\alpha\log\pi$ estimate? How would the target change with $\sigma=0.1$?

**Step 1 — Entropy.** $\ H = \tfrac12\log(2\pi e\,\sigma^2) = \tfrac12\log(2\pi e\cdot0.25) = \tfrac12\log(4.270) = 0.726\ \text{nats}.$

**Step 2 — Soft target.** $\ y = r+\gamma(1-d)\big(\min_i Q_i(s',a') - \alpha\log\pi(a'\mid s')\big) = 1+0.9\big(4.0-0.2(-0.4)\big) = 1+0.9(4.08) = 4.672.$

The entropy term adds $+0.08$ inside the bracket: a random action in a spread-out policy is rewarded.

**Step 3 — Interpretation.** Since $\mathbb E_{a'\sim\pi}[-\log\pi(a'\mid s')] = H(\pi(\cdot\mid s'))$, the bracket estimates the **soft state value** $V(s') = \mathbb E_{a'}[Q(s',a')]+\alpha H(\pi(\cdot\mid s'))$: with $\sigma=0.5$ the entropy contributes $\alpha H = 0.2(0.726)=0.145$ to $V(s')$. With $\sigma=0.1$ the entropy would be $\tfrac12\log(2\pi e\cdot 0.01) = -0.883$ nats (negative: a differential entropy), contributing $-0.177$. A state where the policy must be precise is worth *less* under the maximum-entropy objective than one where it can afford to be random — this is how future entropy flows back into the value of present actions.

**Key concept**

SAC puts the entropy *inside* the return. The soft Bellman target $r+\gamma\big(\min_iQ_{\phi_i^-}(s',a')-\alpha\log\pi(a'\mid s')\big)$ with $a'\sim\pi$ is a one-sample estimate of $r+\gamma V(s')$ where $V$ includes $\alpha H$; states that allow randomness are valued more, and exploration becomes part of what the critic learns.

### Exercise 10.8 — Automatic temperature

SAC adjusts $\alpha$ by gradient descent on $\mathcal L(\alpha) = \mathbb E_{a\sim\pi}\big[-\alpha\big(\log\pi(a\mid s)+\bar H\big)\big]$, with target entropy $\bar H=-d$ for a $d$-dimensional action.

1. Compute $\partial\mathcal L/\partial\alpha$ and express it with the entropy $H$ of the current policy.
2. For $d=1$ ($\bar H=-1$), what happens to $\alpha$ when the current policy has $H=-0.3$? And when $H=-1.8$? Interpret.
3. Why is the optimisation done on $\log\alpha$ rather than on $\alpha$?

**Step 1 — Gradient.** $\ \dfrac{\partial\mathcal L}{\partial\alpha} = -\mathbb E_{a\sim\pi}\big[\log\pi(a\mid s)+\bar H\big] = H\big(\pi(\cdot\mid s)\big)-\bar H,$ using $\mathbb E[\log\pi] = -H$. Gradient *descent* therefore moves $\alpha$ by $-\eta\,(H-\bar H)$.

**Step 2 — Two cases** ($\bar H=-1$).

- $H=-0.3 > \bar H$: the policy is *more random* than the target; $\partial\mathcal L/\partial\alpha = +0.7>0$, so descent **decreases** $\alpha$ — less weight on entropy, the policy is allowed to sharpen.
- $H=-1.8 < \bar H$: the policy is *too deterministic*; $\partial\mathcal L/\partial\alpha = -0.8<0$, so descent **increases** $\alpha$ — more weight on entropy, the policy is pushed to spread out.

The temperature acts as a thermostat keeping the entropy near $\bar H$: high early in training (when a sharpening policy would be premature), lower as the critic becomes reliable.

**Step 3 — Why $\log\alpha$.** The temperature must stay positive, and its useful range spans orders of magnitude (from $\sim 1$ down to $\sim 10^{-3}$). Parameterising $\alpha=e^{\lambda}$ and descending on $\lambda=\log\alpha$ guarantees positivity and makes the step size *relative*: a step in $\lambda$ changes $\alpha$ by a fixed percentage, which is appropriate across the whole range.

**Key concept**

The automatic temperature replaces a hand-tuned entropy coefficient with a feedback loop on the policy's entropy: $\alpha$ rises when the policy is too deterministic and falls when it is too random, relative to the heuristic target $\bar H=-\dim(\mathcal A)$. This is one of the main reasons SAC is robust across environments with very different reward scales.

### Exercise 10.9 — Score-function vs. reparameterised gradient (integrative)

In a fixed state the critic is $Q(a)=-a^2$ (maximum at $a=0$) and the policy is $\pi_\theta(a)=\mathcal N(a;\mu,\sigma^2)$ with $\sigma=1$ fixed and $\mu=1$. We want $\nabla_\mu\,\mathbb E_{a\sim\pi}[Q(a)]$.

1. Compute the exact gradient.
2. Write the **score-function** estimator $\hat g_{\text{SF}} = Q(a)\,\nabla_\mu\log\pi(a)$ and the **reparameterised** estimator $\hat g_{\text{RP}} = \nabla_aQ(a)\big|_{a=\mu+\xi}$ with $\xi\sim\mathcal N(0,1)$; check that both are unbiased.
3. Compare their variances.

**Step 1 — Exact gradient.** $\ \mathbb E[Q(a)] = -\mathbb E[a^2] = -(\mu^2+\sigma^2) = -(1+1) = -2$, so $\ \nabla_\mu\mathbb E[Q] = -2\mu = -2.$

**Step 2 — The two estimators.**

- Score function: $\nabla_\mu\log\pi(a) = (a-\mu)/\sigma^2 = a-1$, so $\hat g_{\text{SF}} = -a^2(a-1)$. With $a=1+\xi$: $\mathbb E[\hat g_{\text{SF}}] = -\mathbb E[(1+\xi)^2\xi] = -\mathbb E[\xi+2\xi^2+\xi^3] = -(0+2+0) = -2.\ \checkmark$
- Reparameterised: $a=\mu+\xi$, $\hat g_{\text{RP}} = \nabla_aQ(a) = -2a = -2(1+\xi)$, so $\mathbb E[\hat g_{\text{RP}}] = -2.\ \checkmark$

**Step 3 — Variances.** $\ \mathrm{Var}[\hat g_{\text{RP}}] = 4\,\mathrm{Var}[\xi] = 4.$ For the score-function estimator, with $\mathbb E[\xi^2]=1,\ \mathbb E[\xi^4]=3,\ \mathbb E[\xi^6]=15$ and odd moments zero:

$\displaystyle \mathbb E\big[\hat g_{\text{SF}}^2\big] = \mathbb E\big[(1+\xi)^4\xi^2\big] = \mathbb E\big[\xi^2+4\xi^3+6\xi^4+4\xi^5+\xi^6\big] = 1+0+18+0+15 = 34, \qquad \mathrm{Var}[\hat g_{\text{SF}}] = 34-4 = 30.$

The reparameterised estimator has **7.5 times less variance** here, because it uses the *gradient* of $Q$ at the sampled action (a smooth, informative quantity) rather than the *value* of $Q$ multiplied by a score (which is large whenever $|Q|$ is large, whatever the direction). For a large offset $c$ added to $Q$ the score-function variance grows like $c^2$ (unless a baseline removes it), while the reparameterised one is unchanged, since it only sees $\nabla_aQ$.

**Key concept**

Both estimators are unbiased, but the reparameterisation trick exploits the differentiability of the critic with respect to the action — exactly what a continuous action space offers — and yields a far lower-variance gradient. It is the stochastic generalisation of the deterministic policy gradient and the reason SAC's actor update is so stable.